# R3 campaign — calibrate along the lever, then a BE-PFM image to calibrate afterwards

**Kernel:** the Igor-side Python (the one that holds `inst`, main thread). Run top to bottom; every
stage cell is self-contained and ends with *bias 0 V, withdraw*. Stop between cells whenever you like.

**What it does**

| cell | stage | writes (under `R3\`) | time |
|---|---|---|---|
| 0 | config: R3 data folder, venv python for the analysis | `setup_r3_log.txt` | – |
| 1 | build / adopt `inst` | – | – |
| 2 | pre-flight: CR1/CR2/CR3 at the free end, walk to the base, reachable span | `01_preflight\preflight_result.json`, `preflight_walk_checkpoint.npz` | 15 min |
| 3 | two-domain bias survey, 8 positions × [−9…9] V × spots 1/2 | `10_bias_survey\domains_bias_checkpoint_500nN_scmpit.npz` | 40 min |
| 4 | V_cpd, P, channel ratio from stage 3 (quasi-static band, all positions) | `10_bias_survey\vcpd_current.json` | s |
| 5 | pick the imaging position: E(x) from the calibrated-lever fit (runs in the analysis venv) | `image_position.json` | 1 min |
| 6 | go to the position, tune (saved), quasi-static 20 kHz reference image | `52_image_be\tune_before_*.txt`, `QImg*.ibw` | 6 min |
| **7** | **BE-PFM frames at 0, V_cpd/2, V_cpd (+ 0 V repeat) — your BE call goes here** | `52_image_be\BE_*` | ~20 min |
| 8 | tune after, second position (optional transfer check) | `52_image_be\...` | 10–25 min |
| 9 | 0 V walk at 8 held-out positions (E(x) transfer check for the map) | `21_wideband_validation\...npz` | 12 min |
| 10 | close-out: repeat the bias sweep at the imaging position, final InvOLS | `80_closeout\closeout_ref_checkpoint.npz` | 12 min |
| 11 | manifest of everything written | `R3_MANIFEST.json` | s |

**Where the data go:** everything under `D:\User Data\Liam\ActiveModeMap\DomainsB_SCMPIT_R3\` (cell 0), in
numbered stage sub-folders like R2. Igor's own save path is repointed to the same folder, so the tune
text files and `.ibw` images land there too. The BE files land wherever your BE code saves them; cell 7
copies or records the path of each frame in `52_image_be\image_be_quant.json`, so the folder is
self-contained. Afterwards copy the whole `DomainsB_SCMPIT_R3` folder into `ActiveModeMap_repo\Data\`
on the laptop (that is `FMM_DATA_ROOT`).

**Standing rules** (unchanged from R1/R2): all COM on the main thread, never `inst.close()`, every stage
puts the bias to 0 V and withdraws in `finally`. Interrupting a cell runs its `finally`.


In [ ]:
# ---- 0. CONFIG ---------------------------------------------------------------------------
import os, sys, json, time, glob, subprocess, traceback
import numpy as np

REPO_DIR    = r'C:\Users\Asylum User\Desktop\STAFF Software\Liam\ActiveModeMap'   # instrument checkout
SCRIPTS_DIR = os.path.join(REPO_DIR, 'scripts', 'scmpit')
R3          = r'D:\User Data\Liam\ActiveModeMap\DomainsB_SCMPIT_R3'                 # <- data folder for this campaign
VENV_PY     = os.path.join(REPO_DIR, '.venv-analysis', 'Scripts', 'python.exe')    # analysis venv (INSTRUMENT_SETUP.md)
PAPER_DIR   = os.path.join(REPO_DIR, 'paper_analysis')

LOAD_NN   = 500.0
DRIVE_V   = 1.0                       # V_ac for every tune, image and BE frame
BIAS_V    = [-9, -6, -3, 0, 3, 6, 9]  # two-domain survey
N_SURVEY  = 8
SPOT_UP, SPOT_DOWN = 1, 2
F_QS      = 20e3                      # quasi-static reference drive
PROBE     = 'stiff'                   # lever calibration used by the analysis (results/lever_calibration_stiff_mass.json)

for p in (SCRIPTS_DIR, REPO_DIR):
    if p not in sys.path: sys.path.insert(0, p)
import scmpit_common as C

if not R3.rstrip('\\/').endswith('_R3'):
    raise SystemExit(f'R3 folder must end in _R3: {R3!r}')
SUB = {k: os.path.join(R3, k) for k in ('01_preflight', '10_bias_survey', '21_wideband_validation',
                                          '52_image_be', '80_closeout')}
for d in [R3] + list(SUB.values()): os.makedirs(d, exist_ok=True)

# repoint the shared module so anything imported from scripts/scmpit writes here, not into R1/R2
C.FILE_LOC = R3
C.PREFLIGHT_JSON = os.path.join(SUB['01_preflight'], 'preflight_result.json')
VCPD_JSON = os.path.join(SUB['10_bias_survey'], 'vcpd_current.json')
IMGPOS_JSON = os.path.join(R3, 'image_position.json')
say = C.stamp

class Stage:
    """with Stage('name', log_path): ... -> tee stdout to a log, always bias 0 V + withdraw at the end."""
    def __init__(self, tag, log, withdraw=True):
        self.tag, self.log, self.withdraw = tag, log, withdraw
    def __enter__(self):
        self.tee = C.Tee(self.log); self.old = sys.stdout; sys.stdout = self.tee; self.t0 = time.time()
        say('=' * 78); say(f'R3  {self.tag}'); say('=' * 78); return self
    def __exit__(self, et, ev, tb):
        if et is not None: say(f'*** FAILED: {et.__name__}: {ev}'); traceback.print_tb(tb)
        if self.withdraw and 'inst' in globals(): C.finish(globals()['inst'], self.t0, self.tag)
        else: say(f'=== {self.tag} finished ({(time.time() - self.t0) / 60:.1f} min) ===')
        sys.stdout = self.old; self.tee.close(); return False

with Stage('SETUP', os.path.join(R3, 'setup_r3_log.txt'), withdraw=False):
    say(f'data folder : {R3}'); say(f'analysis venv: {VENV_PY}  exists={os.path.exists(VENV_PY)}')
    say(f'load {LOAD_NN:.0f} nN, V_ac {DRIVE_V} V, survey biases {BIAS_V} V, spots {SPOT_UP}/{SPOT_DOWN}')


## 1. Instrument
Adopts an existing `inst` if the kernel has one (position bookkeeping is software-tracked — never build a second one blind); otherwise connects to Igor with the laser assumed **at the free end**. Igor's save folder and base name are repointed to R3.

In [ ]:
# ---- 1. BUILD / ADOPT inst ---------------------------------------------------------------
from activemodemap.asylum import AFMLaserSweepAutomation, AsylumInstrument
FORCE_NEW_INST = False

if 'inst' in globals() and not FORCE_NEW_INST:
    say(f'adopting existing inst: x={inst.current_x} um, spot={inst.current_spot}, load={inst.load_nN} nN, '
        f'bias={inst.dc_bias_V:+.1f} V, span={inst.span_um}, limits={inst.x_limits_um}')
else:
    import win32com.client
    igor = win32com.client.Dispatch('IgorPro.Application'); say('connected to Igor Pro')
    automation = AFMLaserSweepAutomation(igor, R3, C.BASE_FILENAME, log_filename=os.path.join(R3, 'SCMPIT_log'))
    automation.eigenmode_center_freq = 275e3; automation.resonance_band_Hz = None
    automation.autowedge_pause = 15.0; automation.wait_for_tune_complete = True
    automation.tune_timeout_s = 600.0; automation.verbose_tune = True
    automation.spot_timeout_s = 90.0; automation.spot_stable_tol_V = 2e-3
    L = C.PROBE_L_UM_FALLBACK
    inst = AsylumInstrument(automation, load_nN=LOAD_NN, dc_bias_V=0.0, recalibrate_each=True, analysis_band_Hz=None,
                            start_at='free_end', span_um=L, x_limits_um=(0.0, L),
                            hw_sign_toward_free_end=C.HW_SIGN_TOWARD_FREE_END, apply_bias=False)
    inst.set_position(L, confirm=True)          # the spot IS at the free end when this runs
    inst.preview_moves([L, L - 10.0])           # second entry must read "toward base"
    say(f'inst built: x = {inst.current_x} um (free end of a {L:.0f} um lever, provisional)')

igor = inst.a.igor
inst.a.file_loc = R3; inst.a.base_filename = C.BASE_FILENAME; inst.a.set_folder()
inst.a.invols_bounds = (1e-8, 5e-5)             # R2 lesson: the old 1e-5 ceiling refused real values
gmv = inst.a.get_gmv()
say(f'Igor saves -> {inst.a.file_loc}; MasterPanel k = {gmv.get("SpringConstant")} N/m (campaign {C.K_LEVER_N_PER_M}), '
    f'InvOLS = {gmv.get("InvOLS")}, setpoint = {gmv.get("DeflectionSetpointVolts")} V')

def pv(k, v): igor.Execute(f'PV("{k}", {v})')
def verify(expected, tol=0.02, abstol=None):
    g, bad = inst.a.get_gmv(), []
    for k, want in expected.items():
        got = g.get(k)
        if got is None: say(f'    {k:<14} not in MasterVariables'); continue
        lim = abstol if abstol is not None else tol * max(abs(want), 1e-12)
        ok = abs(got - want) <= lim; say(f'    {k:<14} asked {want:<12.6g} reads {got:<12.6g} {"ok" if ok else "** MISMATCH"}')
        if not ok: bad.append(k)
    return bad
def _scanning():
    return bool(igor.DataFolder(r'root:packages:MFP3D').Wave('OutWaves').GetTextWavePointValue(0, 0))
def grab(base, timeout_s=1200):
    """Start one down-scan with Igor's base name `base`; wait; return the new .ibw path."""
    inst.a.set_folder()
    igor.Execute(f'root:Packages:MFP3D:Main:Variables:BaseName = "{base}"'); igor.Execute('ARCheckSuffix()')
    pat = os.path.join(inst.a.file_loc, f'{base}*.ibw'); before = {p: os.path.getmtime(p) for p in glob.glob(pat)}
    t0 = time.time(); inst.a.ex('DownScan_0', 'MasterPanel'); time.sleep(10)
    while _scanning() and time.time() - t0 < timeout_s: time.sleep(2)
    if time.time() - t0 >= timeout_s: say('    WARNING: scan timeout')
    time.sleep(3)
    new = [p for p in glob.glob(pat) if p not in before or os.path.getmtime(p) > before[p]]
    if not new: raise RuntimeError(f'no new {base}*.ibw written')
    path = max(new, key=os.path.getmtime); last = -1
    for _ in range(20):
        s = os.path.getsize(path)
        if s == last and s > 0: break
        last = s; time.sleep(0.5)
    say(f'    -> {os.path.basename(path)} ({os.path.getsize(path) / 1e6:.2f} MB, {(time.time() - t0) / 60:.1f} min)')
    return path
def cr1_from_tune(t, lo, hi):
    td = t['tune_data']; fr = np.asarray(td['frequency'], float); am = np.asarray(td['amplitude'], float)
    m = (fr >= lo) & (fr <= hi)
    if not m.any(): raise RuntimeError(f'no tune points in {lo}-{hi} Hz')
    j = np.argmax(am[m]); f1 = float(fr[m][j]); half = am[m][j] / np.sqrt(2)
    a = j
    while a > 0 and am[m][a] > half: a -= 1
    b = j
    while b < m.sum() - 1 and am[m][b] > half: b += 1
    return dict(f_Hz=f1, amp=float(am[m][j]), fwhm_Hz=float(fr[m][b] - fr[m][a]),
                Q=float(f1 / max(fr[m][b] - fr[m][a], 1.0)), snr=float(am[m].max() / max(np.median(am[m]), 1e-12)))
say('helpers ready')


## 2. Pre-flight
Free end first (CR1/CR2/CR3, InvOLS), then walk toward the base in 10 µm steps until the detection fails. The last good position is `x_lo`. The marked-spot image window must be open and the tune panel on the 2 MHz / 10 s window.

In [ ]:
# ---- 2. PRE-FLIGHT -----------------------------------------------------------------------
SPOT, STEP_UM, X_FLOOR_UM, MIN_SNR_dB = 1, 10.0, 30.0, 8.0
CR_BANDS_HZ = None   # set from the free-end tune below (R2's 230-340 kHz prior is WRONG when CR1 sits at 350 kHz)
BAND_HALF = 0.15     # each band = found peak x (1 -/+ 0.15)
def bands_from_free_end(freq, amp):
    pk = C.find_peaks_simple(freq, amp, n=3, min_sep_Hz=150e3, min_snr_dB=8.0, f_min_Hz=100e3)
    if not pk: raise RuntimeError('no resonance peaks found in the free-end tune (>100 kHz, >8 dB)')
    b = {f'cr{i + 1}': ((1 - BAND_HALF) * q['f_Hz'], (1 + BAND_HALF) * q['f_Hz']) for i, q in enumerate(pk)}
    for i in range(len(pk), 3): b[f'cr{i + 1}'] = (np.inf, np.inf)          # missing mode -> empty band
    return b, pk
INVOLS_BOUNDS, INVOLS_JUMP = (1e-8, 5e-5), 3.0
L = float(inst.span_um or C.PROBE_L_UM_FALLBACK)
CKPT = os.path.join(SUB['01_preflight'], 'preflight_walk_checkpoint.npz')

def peaks_in_bands(freq, amp):
    freq = np.asarray(freq, float); amp = np.asarray(amp, float); good = np.isfinite(freq) & np.isfinite(amp) & (amp > 0)
    floor = np.quantile(amp[good & (freq > 50e3)], 0.2); out = []
    for name, (lo, hi) in CR_BANDS_HZ.items():
        m = good & (freq >= lo) & (freq <= hi)
        if not m.any(): out.append(dict(name=name, f_Hz=None, snr_dB=-np.inf, fwhm_Hz=None)); continue
        idx = np.where(m)[0]; j = idx[np.argmax(amp[idx])]; half = amp[j] / np.sqrt(2)
        a = j
        while a > idx[0] and amp[a] > half: a -= 1
        b = j
        while b < idx[-1] and amp[b] > half: b += 1
        out.append(dict(name=name, f_Hz=float(freq[j]), amp=float(amp[j]), snr_dB=float(20 * np.log10(amp[j] / floor)), fwhm_Hz=float(freq[b] - freq[a])))
    return out
def measure(x):
    label = inst._goto_and_prepare(x, capture_image=True); sp = inst.set_load(LOAD_NN, reengage=True)
    freq, Z, rec = inst._tune_and_read(label, sp); inst.a.withdraw(); time.sleep(1.0)
    return freq, Z, rec, peaks_in_bands(freq, np.abs(Z))

xs, Zs, recs, peaks_by_x, freq_ref = [], [], [], {}, None
result = dict(date=time.strftime('%Y-%m-%d %H:%M:%S'), probe='SCM-PIT', k_N_per_m=C.K_LEVER_N_PER_M, f0_free_Hz=C.F0_FREE_HZ,
              load_nN=LOAD_NN, spot=SPOT, probe_L_um=L, x_hi_um=L, x_lo_um=None, step_um=STEP_UM, stop_reason=None,
              cr1_Hz=None, cr2_Hz=None, cr3_Hz=None, cr1_fwhm_Hz=None)
with Stage('PRE-FLIGHT', os.path.join(SUB['01_preflight'], 'preflight_log.txt')):
    try:
        inst.x_limits_um = (0.0, L); inst.load_nN = LOAD_NN; inst.set_dc_bias(0.0); inst.goto_spot(SPOT)
        x = L; say(f'--- free end x = {x:.1f} um ---')
        label = inst._goto_and_prepare(x, capture_image=True); sp = inst.set_load(LOAD_NN, reengage=True)
        freq, Z, rec = inst._tune_and_read(label, sp); inst.a.withdraw(); time.sleep(1.0)
        CR_BANDS_HZ, found = bands_from_free_end(freq, np.abs(Z))
        say('peaks found at the free end: ' + ', '.join(f'{q["f_Hz"]/1e3:.1f} kHz ({q["snr_dB"]:.0f} dB)' for q in found))
        say('bands: ' + ', '.join(f'{k} {lo/1e3:.0f}-{hi/1e3:.0f} kHz' for k, (lo, hi) in CR_BANDS_HZ.items() if np.isfinite(lo)))
        if found[0]['f_Hz'] < 150e3: say('  ** first peak below 150 kHz -- is that CR1 or a torsional/electrostatic feature? check before continuing')
        pk = peaks_in_bands(freq, np.abs(Z)); freq_ref = freq; xs.append(x); Zs.append(Z); recs.append(rec); peaks_by_x[x] = pk
        result['bands_Hz'] = {k: list(v) for k, v in CR_BANDS_HZ.items() if np.isfinite(v[0])}
        say(f'InvOLS {rec["invols_m_per_V"]} m/V, setpoint {rec["setpoint_V"]:.3f} V, {rec["n_tune_points"]} pts')
        for p in pk:
            if p['f_Hz']: say(f'   {p["name"].upper()}: {p["f_Hz"]/1e3:8.2f} kHz  FWHM {p["fwhm_Hz"]/1e3:6.2f} kHz  Q~{p["f_Hz"]/max(p["fwhm_Hz"],1):5.0f}  {p["snr_dB"]:5.1f} dB')
        bn = {p['name']: p for p in pk}
        if bn['cr1']['snr_dB'] < 3 and bn['cr2']['snr_dB'] < 3: raise RuntimeError('neither CR1 nor CR2 visible at the free end')
        result.update(peaks_free_end=pk, cr1_Hz=bn['cr1']['f_Hz'], cr1_fwhm_Hz=bn['cr1']['fwhm_Hz'], cr2_Hz=bn['cr2']['f_Hz'], cr3_Hz=bn['cr3']['f_Hz'])
        last_invols, x_lo, x = rec['invols_m_per_V'], L, L - STEP_UM
        while x >= X_FLOOR_UM - 1e-9:
            say(f'--- x = {x:.1f} um ---')
            try: freq, Z, rec, pk = measure(x)
            except Exception as e: result['stop_reason'] = f'x={x:.1f}: {type(e).__name__}: {e}'; say('STOP: ' + result['stop_reason']); break
            v = rec.get('invols_m_per_V'); bn = {p['name']: p for p in pk}; snr = max(bn['cr1']['snr_dB'], bn['cr2']['snr_dB'])
            say(f'InvOLS {v}, CR1 {(bn["cr1"]["f_Hz"] or 0)/1e3:.2f} kHz {bn["cr1"]["snr_dB"]:.1f} dB, CR2 {(bn["cr2"]["f_Hz"] or 0)/1e3:.2f} kHz {bn["cr2"]["snr_dB"]:.1f} dB, CR3 {(bn["cr3"]["f_Hz"] or 0)/1e3:.2f} kHz')
            if v is None or not (INVOLS_BOUNDS[0] < v < INVOLS_BOUNDS[1]) or (last_invols and max(v / last_invols, last_invols / v) > INVOLS_JUMP):
                result['stop_reason'] = f'x={x:.1f}: InvOLS {v} bad or jumped from {last_invols}'; say('STOP: ' + result['stop_reason']); break
            if snr < MIN_SNR_dB: result['stop_reason'] = f'x={x:.1f}: SNR {snr:.1f} dB < {MIN_SNR_dB}'; say('STOP: ' + result['stop_reason']); break
            xs.append(x); Zs.append(Z); recs.append(rec); peaks_by_x[x] = pk; last_invols = v; x_lo = x; x -= STEP_UM
        else:
            result['stop_reason'] = f'reached X_FLOOR_UM = {X_FLOOR_UM}'
        result.update(x_lo_um=float(x_lo), n_good_positions=len(xs), invols_by_x={f'{xx:.1f}': r['invols_m_per_V'] for xx, r in zip(xs, recs)})
        for nm in ('cr1', 'cr2', 'cr3'):
            result[f'{nm}_by_x'] = {f'{xx:.1f}': next((p['f_Hz'] for p in peaks_by_x[xx] if p['name'] == nm), None) for xx in xs}
        say(f'REACHABLE SPAN: x = {x_lo:.0f} .. {L:.0f} um, {len(xs)} good positions; CR1 {result["cr1_Hz"]/1e3:.2f} kHz')
        inst.x_limits_um = (float(x_lo), float(L))
    finally:
        if xs:
            np.savez(CKPT, x_um=np.array(xs), freq_Hz=np.asarray(freq_ref), Z=np.array(Zs), load_nN=LOAD_NN, spot=SPOT,
                     records=json.dumps(recs, default=str), peaks=json.dumps(peaks_by_x, default=str)); say(f'walk checkpoint -> {CKPT}')
        json.dump(result, open(C.PREFLIGHT_JSON, 'w', encoding='utf-8'), indent=1, default=str); say(f'pre-flight -> {C.PREFLIGHT_JSON}')
if result['x_lo_um'] is None: raise SystemExit('pre-flight did not establish a span')


## 3. Two-domain bias survey (the sparse capture the calibration is built from)
8 equispaced positions over the reachable span, biases interleaved, spots 1 and 2 as the outer loop, free end first. This is the capture `pick_image_position.py` fits and the one the paper's EB arm reconstructs.

In [ ]:
# ---- 3. TWO-DOMAIN BIAS SURVEY -----------------------------------------------------------
from activemodemap import make_conditions, run_series, load_checkpoint
P = C.load_preflight(); C.check_inst(inst, P)
X_SURVEY = np.round(np.linspace(float(P['x_lo_um']), float(P['probe_L_um']), N_SURVEY), 1)
CKPT_SURVEY = os.path.join(SUB['10_bias_survey'], 'domains_bias_checkpoint_500nN_scmpit.npz')
with Stage('STAGE 1  TWO-DOMAIN BIAS SURVEY', os.path.join(SUB['10_bias_survey'], 'bias_domain_survey_500nN_scmpit_log.txt')):
    cond = make_conditions(BIAS_V, [LOAD_NN], spots=[SPOT_UP, SPOT_DOWN], bias_order='interleaved')
    say(f'{len(cond)} conditions/position; positions {X_SURVEY.tolist()} (free end first); checkpoint {CKPT_SURVEY}')
    pv('DriveAmplitude', DRIVE_V); time.sleep(1); verify({'DriveAmplitude': DRIVE_V}, tol=0.002)
    series = run_series(inst, X_SURVEY, cond, ref_index=0, rank=4, dns_band_Hz=C.cr1_band(P),
                        checkpoint_path=CKPT_SURVEY, resume=True, positions_um=X_SURVEY[::-1], verbose=True)
    say(f"{series['x_um'].size} positions x {len(cond)} conditions x {series['freq_Hz'].size} frequencies -- survey done")


## 4. V_cpd, P and the channel ratio from the survey (analysis only)
Quasi-static band, every position (V_cpd should not depend on x; the spread is the check). Writes `vcpd_current.json`, which the BE cell reads for its bias points.

In [ ]:
# ---- 4. V_cpd FROM STAGE 1 ---------------------------------------------------------------
QS_BAND = (15e3, 45e3)
d = np.load(CKPT_SURVEY, allow_pickle=True)
F, Z, x = np.asarray(d['freq_Hz'], float), np.asarray(d['Z']), np.asarray(d['x_um'], float)
cond = json.loads(str(d['conditions'])); bias = np.array([c['bias_V'] for c in cond], float); spot = np.array([c['spot'] for c in cond], int)
s1, s2 = spot == SPOT_UP, spot == SPOT_DOWN
m = (F >= QS_BAND[0]) & (F <= QS_BAND[1]); zq = Z[:, :, m].mean(-1)                       # (cond, pos)
rows = []
for j in range(len(x)):
    (b1, a1) = np.polyfit(bias[s1], zq[s1, j], 1); (b2, a2) = np.polyfit(bias[s2], zq[s2, j], 1)
    Pq, b = (a1 - a2) / 2, (b1 + b2) / 2
    rows.append(dict(x_um=float(x[j]), v_cpd=float(np.real(-(a1 + a2) / (2 * b))), P_abs=float(abs(Pq)), ratio=float(abs(b) / abs(Pq)),
                     flip_deg=float(np.degrees(abs(np.angle(a1) - np.angle(a2))))))
for r in rows: print(f"x {r['x_um']:6.1f}  V_cpd {r['v_cpd']:+6.3f} V  |P|_qs {r['P_abs']:.3e}  |b|/|P| {r['ratio']:.3f}  flip {r['flip_deg']:5.0f} deg")
v = float(np.median([r['v_cpd'] for r in rows])); spread = float(np.ptp([r['v_cpd'] for r in rows]))
if not (-5 < v < 5): raise SystemExit(f'V_cpd {v} not physical')
json.dump(dict(v_cpd_V=v, spread_V=spread, source='R3_stage1_quasistatic_band_median_over_positions', per_position=rows,
               band_Hz=list(QS_BAND), written=time.strftime('%Y-%m-%d %H:%M:%S')), open(VCPD_JSON, 'w'), indent=1)
print(f'\nV_cpd = {v:+.3f} V (spread over positions {spread:.3f} V)  ->  {VCPD_JSON}')


## 5. Pick the imaging position
Runs `paper_analysis\tools\pick_image_position.py` in the analysis venv (Python ≥ 3.10; this kernel is 3.7): fits the calibrated lever to the survey, prints E(x) along the lever and recommends a position with large E that changes < 3 % over ±2 µm of laser drift. Override `X_IMG` by hand if you prefer another spot from the table.

In [ ]:
# ---- 5. PICK THE IMAGING POSITION (analysis venv) ---------------------------------------
if not os.path.exists(VENV_PY): raise SystemExit(f'analysis venv missing: {VENV_PY} -- see INSTRUMENT_SETUP.md')
env = dict(os.environ, FMM_DATA_ROOT=os.path.dirname(R3), FMM_AMM_REPO=REPO_DIR, PYTHONPATH=PAPER_DIR)
r = subprocess.run([VENV_PY, os.path.join('tools', 'pick_image_position.py'), CKPT_SURVEY, '--probe', PROBE, '--tol_pct', '3', '--drift_um', '2',
                    '--preflight', C.PREFLIGHT_JSON],            # bands from THIS campaign's CR frequencies, not the R2 priors
                   cwd=PAPER_DIR, env=env, capture_output=True, text=True)
print(r.stdout); print(r.stderr[-3000:])
pick = json.load(open(os.path.join(PAPER_DIR, 'results', f'pick_image_position_{os.path.splitext(os.path.basename(CKPT_SURVEY))[0]}.json')))
rec = pick['recommended']
X_IMG = float(rec['stage_um']) if rec else None          # <- override here if you want a different stop, e.g. X_IMG = 100.0
E_MAP = {round(t['stage_um']): t['E'] for t in pick['table']}
json.dump(dict(x_img_um=X_IMG, E_at_x=E_MAP.get(round(X_IMG)) if X_IMG else None, c_um=pick['c_um'], f_CR1_Hz=pick['f_CR1_Hz'],
               recommended=rec, picked=time.strftime('%Y-%m-%d %H:%M:%S')), open(IMGPOS_JSON, 'w'), indent=1)
print(f'\nX_IMG = {X_IMG} um (stage), E_map = {E_MAP.get(round(X_IMG)) if X_IMG else None}  ->  {IMGPOS_JSON}')


## 6. Go to the position, tune, quasi-static reference image
Laser to `X_IMG`, AutoWedge + InvOLS, engage at 500 nN, one saved tune (this is the CR1 frequency, linewidth and **Q_tune** the BE band is set from), then a single-frequency 20 kHz reference frame at 0 V (`QImg`), so d33 can be cross-checked the R2 way as well.

In [ ]:
# ---- 6. POSITION, TUNE BEFORE, QS REFERENCE IMAGE ----------------------------------------
FRAME = dict(ScanSize=6e-06, XOffset=7.9308e-07, YOffset=-1.1606e-06, ScanAngle=90.0, ScanPoints=256, ScanLines=256, ScanRate=1.0016)  # PFM_befre0000 frame (R1/R2)
CR1_SEARCH = (0.85 * P['cr1_Hz'], 1.15 * P['cr1_Hz'])
V_CPD = float(json.load(open(VCPD_JSON))['v_cpd_V']); V_HALF = round(V_CPD / 2, 3)
IMG_JSON = os.path.join(SUB['52_image_be'], 'image_be_quant.json')
img = dict(started=time.strftime('%Y-%m-%d %H:%M:%S'), frame=FRAME, load_nN=LOAD_NN, drive_V=DRIVE_V, v_cpd_V=V_CPD, v_half_V=V_HALF,
           x_img_um=X_IMG, positions={}, images=[], be_frames=[])
tune_idx = 900

def goto_and_tune(x, tag):
    """Laser -> x, prepare, engage, saved tune; returns the CR1 peak dict and stores it under positions[x][tag]."""
    global tune_idx
    say(f'=== laser -> stage x = {x:.1f} um, AutoWedge + InvOLS, engage {LOAD_NN:.0f} nN, tune ({tag}) ===')
    inst.a.withdraw(); time.sleep(1); inst._goto_and_prepare(x, capture_image=True)
    if inst._invols is None: raise RuntimeError('InvOLS None at imaging position')
    inst.set_dc_bias(0.0); sp = inst.set_load(LOAD_NN, reengage=True)
    t = inst.a.tune_eigenmode(position_label=f'{tag}_X{x:05.1f}', scan_index=tune_idx, save_tune_data=True); tune_idx += 1
    pk = cr1_from_tune(t, *CR1_SEARCH); pk.update(invols=float(inst._invols), setpoint_V=float(sp), time=time.strftime('%H:%M:%S'))
    say(f'  InvOLS {inst._invols:.4e} m/V, CR1 {pk["f_Hz"]/1e3:.3f} kHz, FWHM {pk["fwhm_Hz"]/1e3:.2f} kHz, Q_tune {pk["Q"]:.0f}, peak/median {pk["snr"]:.1f}')
    if pk['snr'] < 3: say('  ** WARNING: weak CR1 peak -- check the tune window')
    img['positions'].setdefault(str(x), {})[tag] = pk; json.dump(img, open(IMG_JSON, 'w'), indent=1, default=str)
    return pk

def sf_image(name, x, vdc, f_drive, vac=DRIVE_V):
    """One single-frequency frame (Igor scan) with the drive verified; records the path."""
    say(f'--- {name}: V_dc {vdc:+.3f} V, drive {f_drive/1e3:.3f} kHz, {vac} V ---')
    inst.set_dc_bias(vdc); pv('DriveFrequency', f_drive); pv('DriveAmplitude', vac); time.sleep(1)
    if verify({'DriveFrequency': f_drive, 'DriveAmplitude': vac}, tol=0.002): raise RuntimeError('drive did not take')
    path = grab('QImg')
    img['images'].append(dict(name=name, stage_x_um=x, bias_V=vdc, drive_Hz=f_drive, drive_V=vac, invols=float(inst._invols), path=path, time=time.strftime('%H:%M:%S')))
    json.dump(img, open(IMG_JSON, 'w'), indent=1, default=str); return path

with Stage('STAGE 6  POSITION + TUNE BEFORE + QS REFERENCE', os.path.join(SUB['52_image_be'], 'image_be_log.txt'), withdraw=False):
    inst.set_dc_bias(0.0); inst.goto_spot(SPOT_UP)                       # frame where the spots were marked
    for k, v in FRAME.items(): pv(k, v)
    pv('DriveAmplitude', DRIVE_V); time.sleep(1.5)
    bad = verify({k: FRAME[k] for k in ('ScanSize', 'ScanPoints', 'ScanLines', 'ScanRate', 'ScanAngle')}) + verify({k: FRAME[k] for k in ('XOffset', 'YOffset')}, abstol=5e-8)
    if [k for k in bad if k in ('ScanSize', 'XOffset', 'YOffset', 'ScanAngle')]: raise RuntimeError(f'scan frame did not take: {bad}')
    tune_before = goto_and_tune(X_IMG, 'tune_before')
    F_CR1 = tune_before['f_Hz']; FWHM = tune_before['fwhm_Hz']
    sf_image('A_qs20k_0V', X_IMG, 0.0, F_QS)
    say(f'tip is ENGAGED at x = {X_IMG} um, bias 0 V -- go straight on to the BE cell')


## 7. ▶▶ BE-PFM IMAGING — EDIT THIS CELL ◀◀
The tip is engaged at `X_IMG`, 500 nN, 0 V, on the same frame as the reference image. Put **your** BE call inside `be_acquire`. Contract:

* `be_acquire(name, v_dc, v_ac, f_center_Hz, f_width_Hz, out_dir) -> path` — acquires one BE frame with the DC bias already set by this notebook, band centred on `f_center_Hz` with full width `f_width_Hz`, drive `v_ac`, and returns the path of the saved file (copy it into `out_dir` if your code saves elsewhere). Save the **per-pixel spectra**, not only the SHO fits.
* Band: centre = CR1 from `tune_before`, width = 10 × FWHM (clamped to 20–80 kHz), so the resonance and a few linewidths each side are inside the band whatever the pixel-to-pixel shift.
* Pixel time ≳ 4 ms (≥ a few ring-down times of Q ≈ 80 at 290 kHz) — put what you used in `BE_PIXEL_TIME_S` so it is recorded.
* Frames: 0 V, V_cpd/2, V_cpd, 0 V repeat (drift check). Between frames only the bias changes.

The notebook sets the bias and records everything (position, V_dc, V_ac, band, pixel time, InvOLS, tune before/after, file path) in `52_image_be\image_be_quant.json`.

In [ ]:
# ---- 7. BE-PFM FRAMES  >>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT be_acquire()  <<<<<<<<<<<<<<<<<<<<<<<<<<<<
BE_PIXEL_TIME_S = 0.004          # what your BE code actually uses per pixel (recorded, not set here)
BE_WIDTH_HZ = float(np.clip(10 * FWHM, 20e3, 80e3))
BE_CENTER_HZ = F_CR1
BE_OUT = SUB['52_image_be']

def be_acquire(name, v_dc, v_ac, f_center_Hz, f_width_Hz, out_dir):
    """>>> REPLACE THE BODY with your BE call. Must return the saved file path. <<<
    Example shape (adapt names to your BE module):
        from my_be import run_be_image
        path = run_be_image(f_center=f_center_Hz, f_width=f_width_Hz, v_ac=v_ac, pixel_time=BE_PIXEL_TIME_S,
                            frame=FRAME, save_dir=out_dir, base=f'BE_{name}')
        return path
    """
    raise NotImplementedError('put your BE acquisition call here')

BE_PLAN = [('A_be_0V', 0.0), ('A_be_Vhalf', V_HALF), ('A_be_Vcpd', V_CPD), ('A_be_0V_repeat', 0.0)]
with Stage('STAGE 7  BE-PFM FRAMES', os.path.join(SUB['52_image_be'], 'image_be_log.txt'), withdraw=False):
    say(f'band {BE_CENTER_HZ/1e3:.2f} kHz +- {BE_WIDTH_HZ/2e3:.1f} kHz (10 x FWHM {FWHM/1e3:.2f} kHz), V_ac {DRIVE_V} V, pixel {BE_PIXEL_TIME_S*1e3:.1f} ms')
    for name, vdc in BE_PLAN:
        say(f'--- {name}: V_dc {vdc:+.3f} V ---'); inst.set_dc_bias(vdc); time.sleep(1.0)
        t0 = time.time(); path = be_acquire(name, vdc, DRIVE_V, BE_CENTER_HZ, BE_WIDTH_HZ, BE_OUT)
        img['be_frames'].append(dict(name=name, stage_x_um=X_IMG, bias_V=vdc, drive_V=DRIVE_V, f_center_Hz=BE_CENTER_HZ, f_width_Hz=BE_WIDTH_HZ,
                                     pixel_time_s=BE_PIXEL_TIME_S, invols=float(inst._invols), Q_tune_before=tune_before['Q'], path=str(path),
                                     minutes=(time.time() - t0) / 60, time=time.strftime('%H:%M:%S')))
        json.dump(img, open(IMG_JSON, 'w'), indent=1, default=str); say(f'    -> {path}')
    inst.set_dc_bias(0.0); say('BE frames done; tip still engaged for the tune-after')


## 8. Tune after; optional second position
The tune-after bounds the drift of CR1/Q/InvOLS across the BE series. The second position is a **transfer check** of the map (the same d33 must come out from a different M(x)·Q); it costs one QS reference and one BE frame at 0 V. Set `X_IMG2 = None` to skip.

In [ ]:
# ---- 8. TUNE AFTER (+ optional second position) ------------------------------------------
X_IMG2 = None                   # e.g. the 'alternative' from cell 5, or None to skip
with Stage('STAGE 8  TUNE AFTER / SECOND POSITION', os.path.join(SUB['52_image_be'], 'image_be_log.txt')):
    t_after = inst.a.tune_eigenmode(position_label=f'tune_after_X{X_IMG:05.1f}', scan_index=tune_idx, save_tune_data=True); tune_idx += 1
    pk = cr1_from_tune(t_after, *CR1_SEARCH); pk.update(invols=float(inst._invols), time=time.strftime('%H:%M:%S'))
    img['positions'][str(X_IMG)]['tune_after'] = pk
    say(f'CR1 {tune_before["f_Hz"]/1e3:.3f} -> {pk["f_Hz"]/1e3:.3f} kHz, Q_tune {tune_before["Q"]:.0f} -> {pk["Q"]:.0f}, amp x{pk["amp"]/tune_before["amp"]:.2f}')
    if X_IMG2 is not None:
        tb2 = goto_and_tune(X_IMG2, 'tune_before')
        sf_image('B_qs20k_0V', X_IMG2, 0.0, F_QS)
        inst.set_dc_bias(0.0); t0 = time.time()
        path = be_acquire('B_be_0V', 0.0, DRIVE_V, tb2['f_Hz'], float(np.clip(10 * tb2['fwhm_Hz'], 20e3, 80e3)), BE_OUT)
        img['be_frames'].append(dict(name='B_be_0V', stage_x_um=X_IMG2, bias_V=0.0, drive_V=DRIVE_V, f_center_Hz=tb2['f_Hz'], f_width_Hz=float(np.clip(10 * tb2['fwhm_Hz'], 20e3, 80e3)),
                                     pixel_time_s=BE_PIXEL_TIME_S, invols=float(inst._invols), Q_tune_before=tb2['Q'], path=str(path), minutes=(time.time() - t0) / 60, time=time.strftime('%H:%M:%S')))
        ta2 = inst.a.tune_eigenmode(position_label=f'tune_after_X{X_IMG2:05.1f}', scan_index=tune_idx, save_tune_data=True); tune_idx += 1
        img['positions'][str(X_IMG2)]['tune_after'] = cr1_from_tune(ta2, *CR1_SEARCH)
    img['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); json.dump(img, open(IMG_JSON, 'w'), indent=1, default=str); say(f'-> {IMG_JSON}')


## 9. Held-out 0 V walk (E(x) transfer check)
8 positions at the midpoints between the survey positions, spot 1, 0 V, 500 nN — every one is half a pitch from any position the map was built from. Scores the reconstructed E(x) where it was not measured, within the hour of the images.

In [ ]:
# ---- 9. HELD-OUT 0 V WALK ---------------------------------------------------------------
from activemodemap.series import make_conditions as _mc, run_series as _rs
X_VAL = np.round(0.5 * (X_SURVEY[:-1] + X_SURVEY[1:]), 1)[::-1]            # midpoints, free end -> base
CKPT_VAL = os.path.join(SUB['21_wideband_validation'], f'wideband_valid_{X_VAL.size}pos_500nN_0V_checkpoint.npz')
with Stage('STAGE 9  HELD-OUT 0 V WALK', os.path.join(SUB['21_wideband_validation'], 'wideband_valid_log.txt')):
    pv('DriveAmplitude', DRIVE_V); time.sleep(1); verify({'DriveAmplitude': DRIVE_V}, tol=0.002)
    res = _rs(inst, X_VAL, _mc([0.0], [LOAD_NN], spots=[SPOT_UP]), positions_um=X_VAL, checkpoint_path=CKPT_VAL, resume=True, verbose=True)
    say(f"captured {len(res['x_um'])} positions x {len(res['freq_Hz'])} frequencies -> {CKPT_VAL}")


## 10. Close-out
The stage-3 recipe repeated at the imaging position only (both spots, all biases): end-of-campaign V_cpd, P and channel ratio to difference against the morning's, plus a final InvOLS. Then withdraw.

In [ ]:
# ---- 10. CLOSE-OUT -----------------------------------------------------------------------
CKPT_CLOSE = os.path.join(SUB['80_closeout'], 'closeout_ref_checkpoint.npz')
with Stage('STAGE 10  CLOSE-OUT', os.path.join(SUB['80_closeout'], 'closeout_log.txt')):
    pv('DriveAmplitude', DRIVE_V); time.sleep(1); verify({'DriveAmplitude': DRIVE_V}, tol=0.002)
    cond = make_conditions(BIAS_V, [LOAD_NN], spots=[SPOT_UP, SPOT_DOWN], bias_order='interleaved')
    res = run_series(inst, np.array([X_IMG]), cond, positions_um=np.array([X_IMG]), checkpoint_path=CKPT_CLOSE, resume=True, verbose=True)
    Fz, Zz = res['freq_Hz'], res['Z']; m = (Fz >= QS_BAND[0]) & (Fz <= QS_BAND[1]); zq = Zz[:, 0, m].mean(-1)
    (b1, a1) = np.polyfit(bias[s1], zq[s1], 1); (b2, a2) = np.polyfit(bias[s2], zq[s2], 1)
    vc = float(np.real(-(a1 + a2) / (2 * (b1 + b2) / 2)))
    inst.a.withdraw(); time.sleep(1); inst._goto_and_prepare(X_IMG, capture_image=True)
    close = dict(x_um=X_IMG, v_cpd_V=vc, v_cpd_morning_V=V_CPD, P_abs_qs=float(abs(a1 - a2) / 2), invols_final=float(inst._invols),
                 spring_panel=inst.a.get_gmv().get('SpringConstant'), time=time.strftime('%Y-%m-%d %H:%M:%S'))
    json.dump(close, open(os.path.join(SUB['80_closeout'], 'closeout_cal.json'), 'w'), indent=1)
    say(f'V_cpd {V_CPD:+.3f} -> {vc:+.3f} V at x = {X_IMG} um; final InvOLS {inst._invols:.4e} m/V')


## 11. Manifest

In [ ]:
# ---- 11. MANIFEST ------------------------------------------------------------------------
files = sorted(os.path.relpath(p, R3) for p in glob.glob(os.path.join(R3, '**', '*'), recursive=True) if os.path.isfile(p))
man = dict(campaign='DomainsB_SCMPIT_R3', root=R3, written=time.strftime('%Y-%m-%d %H:%M:%S'), probe=PROBE, load_nN=LOAD_NN, drive_V=DRIVE_V,
           x_img_um=X_IMG, x_img2_um=X_IMG2, v_cpd_V=V_CPD, preflight=C.PREFLIGHT_JSON, survey=CKPT_SURVEY, image_json=IMG_JSON, files=files)
json.dump(man, open(os.path.join(R3, 'R3_MANIFEST.json'), 'w'), indent=1, default=str)
print(f'{len(files)} files under {R3}'); print('\n'.join(files))


## After the campaign
Copy `DomainsB_SCMPIT_R3\` into `ActiveModeMap_repo\Data\` on the laptop. The calibration then goes: E(x) from the survey with the calibrated EB arm (`tools\pick_image_position.py` already gives the table), Q_tune from `tune_before`, M(x) = E/Q_tune; per BE pixel d33 = A_SHO / (M(x)·Q_pixel) · InvOLS / 32 / V_ac, with the Q_pixel-vs-d33 scatter as the internal check and the QS 20 kHz frame as the model-free cross-check. The BE reader is written once the file format is known (`image_be_quant.json` records every path).